# 15 - SetFit-style contrastive fine-tuning (post-hoc capacity control)

**Question.** Your deployed model uses a *frozen* MiniLM embedder + a logistic-regression head, and you argue the performance ceiling on the three overlapping editorial categories is a **construct problem (the labels), not a capacity problem (the model)**. This notebook tests that directly: if we *let the embedder move* (contrastively fine-tune it on the labels, the SetFit mechanism), does the **triangle** finally separate?

**Hypothesis (yours).** Fine-tuning sharpens the *concrete* categories that are already separable, but does **not** rescue the contested editorial triangle - because the limit is in the taxonomy, not the model.

**Protocol (keep it honest).**
- Train on `data/modelling/train.csv` (942), evaluate on `data/modelling/val.csv` (167) - the SAME val set behind your 0.750 baseline.
- Nothing is fitted on the held-out test set (we add that in step 2 / NB16).
- We re-fit the frozen baseline *in this notebook* so the comparison uses identical metric code.

**Triangle** (overlapping, contested): `policy_practice_research`, `political_environment_key_organisations`, `what_matters_ed`.  
**Concrete** (should be separable): `edtech`, `four_nations`, `teacher_rrd`.

> Why not the `setfit` library? It is incompatible with this repo's `transformers` 5.5.0. We use `sentence-transformers` directly with `BatchAllTripletLoss`, which is the same contrastive mechanism SetFit wraps.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments, losses)
from datasets import Dataset

ROOT = Path.cwd() if (Path.cwd()/'data').exists() else Path.cwd().parent
DATA = ROOT/'data'/'modelling'
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
TEXT_COL, LABEL_COL = 'text_clean', 'target'
SEED = 42

TRIANGLE = ['policy_practice_research','political_environment_key_organisations','what_matters_ed']
CONCRETE = ['edtech','four_nations','teacher_rrd']

train = pd.read_csv(DATA/'train.csv'); val = pd.read_csv(DATA/'val.csv')
CLASSES = sorted(train[LABEL_COL].unique())
print('train', train.shape, '| val', val.shape)
print('classes:', CLASSES)
print(train[LABEL_COL].value_counts().to_string())

In [ ]:
# --- metric helpers (identical for both models) ---
def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, labels=CLASSES, average=None, zero_division=0)
    return dict(zip(CLASSES, f))

def top2_acc(clf, X, y_true):
    proba = clf.predict_proba(X)
    top2 = clf.classes_[np.argsort(proba, axis=1)[:, -2:]]
    return float(np.mean([yt in row for yt, row in zip(y_true, top2)]))

def summarise(name, y_true, y_pred, clf, X):
    pcf = per_class_f1(y_true, y_pred)
    out = {
        'model': name,
        'overall_macroF1': float(np.mean(list(pcf.values()))),
        'triangle_macroF1': float(np.mean([pcf[c] for c in TRIANGLE])),
        'concrete_macroF1': float(np.mean([pcf[c] for c in CONCRETE])),
        'top2_acc': top2_acc(clf, X, y_true),
    }
    out.update({f'F1[{c}]': round(pcf[c],3) for c in CLASSES})
    return out

## 1. Baseline - frozen embedder + logistic-regression head
This reproduces your deployed design. Expect overall macro F1 around 0.75 on val.

In [ ]:
base_model = SentenceTransformer(MODEL_NAME)
Xtr = base_model.encode(train[TEXT_COL].tolist(), show_progress_bar=True)
Xva = base_model.encode(val[TEXT_COL].tolist(), show_progress_bar=True)

base_clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
base_clf.fit(Xtr, train[LABEL_COL])
base_pred = base_clf.predict(Xva)
base_row = summarise('frozen + logreg (baseline)', val[LABEL_COL].values, base_pred, base_clf, Xva)
print('overall macro F1 :', round(base_row['overall_macroF1'],3))
print('triangle macro F1:', round(base_row['triangle_macroF1'],3))
print('concrete macro F1:', round(base_row['concrete_macroF1'],3))
print('top-2 accuracy   :', round(base_row['top2_acc'],3))

## 2. SetFit-style contrastive fine-tune of the embedder, then re-fit the head
`BatchAllTripletLoss` pulls same-label articles together and pushes different-label ones apart, reshaping the 384-dim space using your labels. Then we fit a fresh logreg head on the *fine-tuned* embeddings.

~4-5 min on CPU for 3 epochs. Lower `EPOCHS` to 1 if you want a quick pass first.

In [ ]:
from sentence_transformers import SentenceTransformer as _ST
EPOCHS = 3
c2i = {c:i for i,c in enumerate(CLASSES)}

ft_model = _ST(MODEL_NAME)   # fresh copy so the baseline above is untouched
train_ds = Dataset.from_dict({'sentence': train[TEXT_COL].tolist(),
                              'label': [c2i[t] for t in train[LABEL_COL]]})
loss = losses.BatchAllTripletLoss(ft_model)
args = SentenceTransformerTrainingArguments(
    output_dir='/tmp/setfit_ft', num_train_epochs=EPOCHS,
    per_device_train_batch_size=32, warmup_steps=0.1,
    fp16=False, seed=SEED, report_to=[], logging_steps=50)
SentenceTransformerTrainer(model=ft_model, args=args, train_dataset=train_ds, loss=loss).train()
print('fine-tune done')

In [ ]:
Xtr_ft = ft_model.encode(train[TEXT_COL].tolist(), show_progress_bar=True)
Xva_ft = ft_model.encode(val[TEXT_COL].tolist(), show_progress_bar=True)
ft_clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
ft_clf.fit(Xtr_ft, train[LABEL_COL])
ft_pred = ft_clf.predict(Xva_ft)
ft_row = summarise('contrastive fine-tune + logreg', val[LABEL_COL].values, ft_pred, ft_clf, Xva_ft)
print('overall macro F1 :', round(ft_row['overall_macroF1'],3))
print('triangle macro F1:', round(ft_row['triangle_macroF1'],3))
print('concrete macro F1:', round(ft_row['concrete_macroF1'],3))
print('top-2 accuracy   :', round(ft_row['top2_acc'],3))

## 3. Side by side
The number that matters is **triangle_macroF1**: does fine-tuning move it, or only the concrete categories?

In [ ]:
cmp = pd.DataFrame([base_row, ft_row]).set_index('model')
view = cmp[['overall_macroF1','triangle_macroF1','concrete_macroF1','top2_acc']].round(3)
print(view.to_string())
delta = (ft_row['triangle_macroF1'] - base_row['triangle_macroF1'])
print(f'\nTriangle change from fine-tuning: {delta:+.3f}')
(ROOT/'outputs').mkdir(exist_ok=True)
cmp.round(4).to_csv(ROOT/'outputs'/'setfit_vs_baseline_val.csv')
print('saved -> outputs/setfit_vs_baseline_val.csv')

## 4. Your read first
Before I give mine, what do you make of it? Things to look at:
- Did **overall** macro F1 move much vs your 0.750?
- Did the **concrete** trio improve (separable categories sharpening, as expected)?
- Crucially, did the **triangle** trio improve, stay flat, or drop? Flat/no-gain here is the evidence that the ceiling is the *labels*, not the model.
- Did **top-2** stay high (right answer still in the top two even when the single-label call is contested)?

**Caveats to keep this honest:** this is one run on the val set; contrastive training has seed variance, so a small wiggle is not a real gain. The clean confirmation is the held-out test (step 2) and the same pattern under LoRA (NB16).

_Next: NB16 - LoRA fine-tune of a small encoder (the PEFT skill + a second, independent capacity control)._